# Libraries

In [0]:
# Installing new libraries
%pip install -r ../requirements.txt
%restart_python

In [0]:
# Importing required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.transformers import geometric_adstock, logistic_saturation
from pymc_marketing.prior import Prior
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error

mlflow.autolog(disable=True)

# Data

In [0]:
directory = "/Volumes/training_feature_development/marketing_mix_modelling/data_v2/model_data/"

control_df = pd.read_csv(directory + "exh_control_data.csv", index_col="week")
response_df = pd.read_csv(directory + "exh_response_data.csv", index_col="week")
spend_df = pd.read_csv(directory + "exh_spend_data.csv")

In [0]:
# Processing data
df = spend_df[spend_df["lob"]!="Other"].copy()
df = df.groupby(["week", "lob"]).agg({"spend": "sum"}).unstack().fillna(0)
df.columns = ["brand", "cineclub", "event", "exhibition", 
               "f&b", "film", "g&e", "imax", 
               "pr", "performance", "prepaid"]
df.index.name = "week"

df = pd.concat([df, control_df, response_df], axis=1).reset_index()
df["week"] = pd.to_datetime(df["week"])

# Linear Regression

In [0]:
X = df[df.columns[1:-4]]
y = df["attendance"]

In [0]:
# Setting up and fitting the model
for i in X.columns[:-4]:
    X[i] = MinMaxScaler().fit_transform(X[[i]]).flatten()
    X[i] = geometric_adstock(X[i], alpha=0.5, l_max=4, normalize=True).eval().flatten()
    X[i] = logistic_saturation(X[i], lam=5).eval().flatten()
model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

In [0]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=120)

plt.plot(y_pred, color="tab:blue", label="Predicted Attendance")
plt.plot(y, color="grey", alpha=0.5, label="Actual Attendance")

# Changing aesthetics
plt.ylim(0, 3*10**6)
plt.xlabel("Week Number")
plt.ylabel("Attendance")
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=""))
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.show()

# Outputting metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mape = mean_absolute_percentage_error(y, y_pred)

print(f"R2: {r2:.3f}")
print(f"RMSE: {rmse:.0f}")
print(f"MAPE: {mape:.3f}")

# Marketing Mix Model

In [0]:
X = df[df.columns[:-4]]
y = df["attendance"]

In [0]:
# Setting assumptions
custom_config = {
    # Non-negative baseline sales
    "intercept": Prior("Exponential", lam=1),
    # Setting a non-negative control effect
    "gamma_control": Prior("HalfNormal", sigma=Prior("Gamma", alpha=4, beta=1)),
    # Setting an average retention rate of 40% 
    "adstock_alpha": Prior("Beta", alpha=2, beta=3),
    # Setting a low ceiling for spend
    "saturation_lam": Prior("Gamma", alpha=5, beta=1)
}

# Setting up and fitting the model
mmm = MMM(model_config=custom_config,
          date_column="week",
          channel_columns=X.columns[1:-4],
          control_columns=X.columns[-4:],
          # Maximum effect over 3 weeks
          adstock=GeometricAdstock(l_max=4),
          saturation=LogisticSaturation())
mmm.fit(X, y, chains=2, random_seed=12, target_accept=0.95)

mmm.sample_prior_predictive(X, extend_idata=True, combined=True)
mmm.sample_posterior_predictive(X, extend_idata=True, combined=True)

y_pred = mmm.predict(X)

In [0]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=120)

plt.plot(X["week"], y_pred, color="tab:blue", label="Predicted Attendance")
plt.plot(X["week"], y.reset_index(drop=True), color="grey", alpha=0.5, label="Actual Attendance")

# Changing aesthetics
plt.ylim(0, 3*10**6)
plt.xlabel("Week Number")
plt.ylabel("Attendance")
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=""))
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.show()

# Outputting metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mape = mean_absolute_percentage_error(y, y_pred)

print(f"R2: {r2:.3f}")
print(f"RMSE: {rmse:.0f}")
print(f"MAPE: {mape:.3f}")

In [0]:
mmm.plot_waterfall_components_decomposition();

In [0]:
fig = mmm.plot_channel_contribution_grid(start=0.5, stop=2, num=16)
for collection in fig.gca().collections:
    collection.remove()
fig.set_size_inches(8, 6)
fig.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=""))
fig.gca().set_yticks(np.arange(0, 2.1*10**7, 2*10**6))
fig.gca().set_ylim(0, 2*10**7)
plt.grid(alpha=0.5);

In [0]:
fig = mmm.plot_direct_contribution_curves(xlim_max=1.5, same_axes=True, show_fit=True, quantile_lower=0.5, quantile_upper=0.5);
fig.set_size_inches(9, 6)
fig.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=""))
fig.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=""))
plt.legend(bbox_to_anchor=(1.5, 1), loc="upper right")
plt.grid(alpha=0.5)

# Marketing Mix Model (Awareness)

In [0]:
# Processing data
df = spend_df[(spend_df["objective"]=="Awareness")&(spend_df["lob"]!="Other")].copy()
df = df.groupby(["week", "lob"]).agg({"spend": "sum", "impressions": "sum"}).unstack().fillna(0)
df.columns = ["brand", "cineclub", "f&b", "film", "imax", 
               "brand_impr", "cineclub_impr", "f&b_impr", "film_impr", "imax_impr"]
df.index.name = "week"

df = pd.merge(df, control_df, how="right", left_index=True, right_index=True)
df = pd.concat([df, response_df[["attendance"]]], axis=1)
df = df.fillna(0).reset_index()
df["week"] = pd.to_datetime(df["week"])
df["impressions"] = df.filter(like="impr").sum(axis=1)

In [0]:
X = df[list(df.columns[:6])+list(df.columns[-6:-2])]
y = df["impressions"]

In [0]:
# Setting assumptions
custom_config = {
    # Non-negative baseline sales
    "intercept": Prior("Exponential", lam=4),
    # Setting a non-negative control effect
    "gamma_control": Prior("HalfNormal", sigma=Prior("Gamma", alpha=4, beta=1)),
    # Setting an average retention rate of 40% 
    "adstock_alpha": Prior("Beta", alpha=2, beta=3),
    # Setting a low ceiling for spend
    "saturation_lam": Prior("Gamma", alpha=5, beta=1)
}

# Setting up and fitting the model
mmm = MMM(model_config=custom_config,
          date_column="week",
          channel_columns=X.columns[1:-4],
          control_columns=X.columns[-4:],
          # Maximum effect over 4 weeks
          adstock=GeometricAdstock(l_max=4),
          saturation=LogisticSaturation())
mmm.fit(X, y, chains=2, random_seed=12, target_accept=0.95)

mmm.sample_prior_predictive(X, extend_idata=True, combined=True)
mmm.sample_posterior_predictive(X, extend_idata=True, combined=True)

y_pred = mmm.predict(X)

In [0]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=120)

plt.plot(X["week"], y_pred, color="tab:blue", label="Predicted Impressions")
plt.plot(X["week"], y, color="grey", alpha=0.5, label="Actual Impressions")

# Changing aesthetics
plt.ylim(0, 1.5*10**7)
plt.xlabel("Week Number")
plt.ylabel("Impressions")
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=""))
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.show()

# Outputting metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mape = mean_absolute_percentage_error(y, y_pred)

print(f"R2: {r2:.3f}")
print(f"RMSE: {rmse:.0f}")
print(f"MAPE: {mape:.3f}")

# Marketing Mix Model (Consideration)

In [0]:
# Processing data
df = spend_df[(spend_df["objective"]=="Consideration")&(spend_df["lob"]!="Other")].copy()
df = df.groupby(["week", "lob"]).agg({"spend": "sum", "clicks": "sum"}).unstack().fillna(0)
df.columns = ["brand", "exhibition", "film", "g&e", "imax", "performance", "prepaid",
               "brand_clicks", "exhibition_clicks", "film_clicks", "g&e_clicks", 
               "imax_clicks", "performance_clicks", "prepaid_clicks"]
df.index.name = "week"

df = pd.merge(df, control_df, how="right", left_index=True, right_index=True)
df = pd.concat([df, response_df[["attendance"]]], axis=1)
df = df.fillna(0).reset_index()
df["week"] = pd.to_datetime(df["week"])
df["clicks"] = df.filter(like="clicks").sum(axis=1)

In [0]:
X = df[list(df.columns[:8])+list(df.columns[-6:-2])]
y = df["clicks"]

In [0]:
# Setting assumptions
custom_config = {
    # Non-negative baseline sales
    "intercept": Prior("Exponential", lam=4),
    # Setting a non-negative control effect
    "gamma_control": Prior("HalfNormal", sigma=Prior("Gamma", alpha=4, beta=1)),
    # Setting an average retention rate of 40% 
    "adstock_alpha": Prior("Beta", alpha=2, beta=3),
    # Setting a low ceiling for spend
    "saturation_lam": Prior("Gamma", alpha=5, beta=1)
}

# Setting up and fitting the model
mmm = MMM(model_config=custom_config,
          date_column="week",
          channel_columns=X.columns[1:-4],
          control_columns=X.columns[-4:],
          # Maximum effect over 4 weeks
          adstock=GeometricAdstock(l_max=4),
          saturation=LogisticSaturation())
mmm.fit(X, y, chains=2, random_seed=12, target_accept=0.95)

mmm.sample_prior_predictive(X, extend_idata=True, combined=True)
mmm.sample_posterior_predictive(X, extend_idata=True, combined=True)

y_pred = mmm.predict(X)

In [0]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=120)

plt.plot(X["week"], y_pred, color="tab:blue", label="Predicted Clicks")
plt.plot(X["week"], y, color="grey", alpha=0.5, label="Actual Clicks")

# Changing aesthetics
plt.ylim(0, 2*10**5)
plt.xlabel("Week Number")
plt.ylabel("Clicks")
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=""))
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.show()

# Outputting metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mape = mean_absolute_percentage_error(y, y_pred)

print(f"R2: {r2:.3f}")
print(f"RMSE: {rmse:.0f}")
print(f"MAPE: {mape:.3f}")